In [5]:
import tensorflow as tf

def count_records(file_path):
    return sum(1 for _ in tf.data.TFRecordDataset(file_path))

# Check your splits
print(f"Train count: {count_records('../data/Damaged-package-2/train/package.tfrecord')}")
print(f"Validation count: {count_records('../data/Damaged-package-2/valid/package.tfrecord')}")
print(f"Test count: {count_records('../data/Damaged-package-2/test/package.tfrecord')}")

Train count: 288
Validation count: 27
Test count: 14


In [ ]:
import tensorflow as tf

raw_dataset = tf.data.TFRecordDataset('../data/Damaged-package-2/train/package.tfrecord')
for raw_record in raw_dataset.take(1):
    example = tf.train.Example()
    example.ParseFromString(raw_record.numpy())
    print("Keys found in your TFRecord:")
    for key in example.features.feature.keys():
        print(f"- {key}")

Keys found in your TFRecord:
- image/object/class/text
- image/object/bbox/ymin
- image/object/class/label
- image/height
- image/object/bbox/xmin
- image/width
- image/object/bbox/ymax
- image/filename
- image/format
- image/encoded
- image/object/bbox/xmax


In [ ]:
import tensorflow as tf
from collections import Counter

def get_class_distribution(file_path):
    label_counts = Counter()
    
    feature_description = {
        'image/object/class/label': tf.io.VarLenFeature(tf.int64),
    }

    def _parse_label(proto):
        parsed_features = tf.io.parse_single_example(proto, feature_description)
        # convert the list of labels into array
        labels = tf.sparse.to_dense(parsed_features['image/object/class/label'])
        return labels

    dataset = tf.data.TFRecordDataset(file_path)
    for labels in dataset.map(_parse_label):
        # count every individual object found in the images
        for l in labels.numpy():
            label_counts[l] += 1
        
    return label_counts

id_map = {1: "Minor Damage (Damaged)", 2: "Severe Damage (Destroyed)", 3: "Intact"}

train_path = '../data/Damaged-package-2/train/package.tfrecord'
distribution = get_class_distribution(train_path)

print("Instance Distribution in Training Set:")
for label_id, count in distribution.items():
    print(f"{id_map.get(label_id, 'Unknown')}: {count} instances")

Instance Distribution in Training Set:
Severe Damage (Destroyed): 75 instances
Minor Damage (Damaged): 294 instances
Intact: 782 instances


In [9]:
def parse_to_classification(proto):
    # match keys found 
    feature_description = {
        'image/encoded': tf.io.FixedLenFeature([], tf.string),
        'image/object/class/label': tf.io.VarLenFeature(tf.int64),
    }
    
    parsed = tf.io.parse_single_example(proto, feature_description)
    
    # decode and resize image
    image = tf.io.decode_jpeg(parsed['image/encoded'])
    image = tf.image.resize(image, [224, 224])
    image = image / 255.0  # normalization
    
    # apply logic to determine global label
    labels = tf.sparse.to_dense(parsed['image/object/class/label'])
    
    # 2 (Destroyed) > 1 (Damaged) > 3 (Intact)
    if tf.reduce_any(labels == 2):
        global_label = 2  # severe
    elif tf.reduce_any(labels == 1):
        global_label = 1  # minor
    else:
        global_label = 0  # intact
        
    return image, global_label

# create training pipeline
train_path = '../data/Damaged-package-2/train/package.tfrecord'
train_ds = tf.data.TFRecordDataset(train_path).map(parse_to_classification)
train_ds = train_ds.shuffle(100).batch(32).prefetch(tf.data.AUTOTUNE)